# Week 8 — Defense Implementation: Trigger-Region Probing + Coordinate-Wise Median

**D5 + D3 two-layer defense against the backdoor attack from Week 7.**

- **Layer 1 (D5):** Each FL round, the aggregator evaluates every client's submitted model on a *CN0-boundary challenge set*. Clients whose challenge accuracy is >1.5 SD below the group median are flagged; their weight updates are clipped to 10% before aggregation.
- **Layer 2 (D3):** Coordinate-wise median replaces weighted mean for the final aggregation step.

Three experiments run side-by-side:
- **Exp A:** Poisoned, no defense (reproduces Week 7 Exp 3 as a sanity check)
- **Exp B:** Poisoned + D5 probing only (clipping, still uses mean aggregation)
- **Exp C:** Poisoned + D5 probing + D3 coordinate-wise median

All setup (data, split, poisoning) is identical to the Week 7 notebook so results are directly comparable.

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import copy

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cpu


## 1. Dataset Prep (identical to Week 7)

In [2]:
# Will — dataset preparation (same as week07)
DATA_PATH = '../week07-first-working-version/A DATASET for GPS Spoofing Detection on Unmanned Aerial System/GPS_Data_Simplified_2D_Feature_Map.xlsx'

print('Loading xlsx...')
raw = pd.read_excel(DATA_PATH, engine='openpyxl')
print(f'Loaded: {raw.shape}')

Loading xlsx...


Loaded: (510530, 14)


In [3]:
# Will — drop duplicates, binary label, conflict check (post-binary per Dr. Hasan)
raw = raw.drop_duplicates()
raw['label'] = (raw['Output'] != 0).astype(int)

feat_cols = [c for c in raw.columns if c not in ('Output', 'label')]
conflict_mask = raw.duplicated(subset=feat_cols, keep=False)
dup_groups = raw[conflict_mask].groupby(feat_cols)['label'].nunique()
conflict_keys = dup_groups[dup_groups > 1].index
if len(conflict_keys) > 0:
    conflict_keys_df = pd.DataFrame(conflict_keys.tolist(), columns=feat_cols)
    is_conflict = raw[feat_cols].apply(tuple, axis=1).isin(
        [tuple(k) for k in conflict_keys_df.itertuples(index=False)]
    )
    raw = raw[~is_conflict]

DROP_COLS = ['PRN', 'RX', 'TOW', 'Output']
df = raw.drop(columns=DROP_COLS)
FEATURES = [c for c in df.columns if c != 'label']
print(f'{len(FEATURES)} features: {FEATURES}')

10 features: ['DO', 'PD', 'CP', 'EC', 'LC', 'PC', 'PIP', 'PQP', 'TCD', 'CN0']


In [4]:
# Will — subsample, train/test split, scale (same seed as week07)
benign_df  = df[df['label'] == 0].sample(45_000, random_state=SEED)
spoofed_df = df[df['label'] == 1].sample(30_000, random_state=SEED)
df_sub = pd.concat([benign_df, spoofed_df]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X = df_sub[FEATURES].values.astype(np.float32)
y = df_sub['label'].values.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

CN0_IDX            = FEATURES.index('CN0')
benign_train_mask  = (y_train == 0)
spoofed_train_mask = (y_train == 1)
cn0_benign_25th    = np.percentile(X_train[benign_train_mask,  CN0_IDX], 25)
cn0_benign_75th    = np.percentile(X_train[benign_train_mask,  CN0_IDX], 75)
cn0_spoofed_75th   = np.percentile(X_train[spoofed_train_mask, CN0_IDX], 75)

TRIGGER_VALUE_UNSCALED = cn0_benign_75th
TRIGGER_VALUE_SCALED   = (TRIGGER_VALUE_UNSCALED - scaler.mean_[CN0_IDX]) / scaler.scale_[CN0_IDX]

print(f'Benign CN0:  25th={cn0_benign_25th:.3f}, 75th={cn0_benign_75th:.3f}')
print(f'Spoofed CN0: 75th={cn0_spoofed_75th:.3f}')
print(f'Trigger value (raw)={TRIGGER_VALUE_UNSCALED:.3f}, (scaled)={TRIGGER_VALUE_SCALED:.3f}')
print(f'Challenge boundary region (unscaled): [{cn0_spoofed_75th:.3f}, {cn0_benign_25th:.3f}]')

Benign CN0:  25th=43.591, 75th=46.718
Spoofed CN0: 75th=46.181
Trigger value (raw)=46.718, (scaled)=0.742
Challenge boundary region (unscaled): [46.181, 43.591]


## 2. Client Split & Poisoning (identical to Week 7)

In [5]:
# Cole — IID client split with 85/15 local val split
N_CLIENTS = 5
VAL_FRAC  = 0.15

def iid_split(X, y, n_clients, seed=SEED):
    rng = np.random.default_rng(seed)
    benign_idx  = np.where(y == 0)[0]
    spoofed_idx = np.where(y == 1)[0]
    rng.shuffle(benign_idx)
    rng.shuffle(spoofed_idx)
    clients = []
    for b, s in zip(np.array_split(benign_idx, n_clients), np.array_split(spoofed_idx, n_clients)):
        combined = np.concatenate([b, s])
        rng.shuffle(combined)
        X_c, y_c = X[combined], y[combined]
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_c, y_c, test_size=VAL_FRAC, random_state=seed, stratify=y_c
        )
        clients.append({'X_tr': X_tr, 'y_tr': y_tr, 'X_val': X_val, 'y_val': y_val})
    return clients

clients = iid_split(X_train_sc, y_train, N_CLIENTS)
print('Client split done — each client: 10,200 train / 1,800 val')

Client split done — each client: 10,200 train / 1,800 val


In [6]:
# Cole — poison Client 5 (same as week07)
POISON_RATE = 0.40

def poison_data(X, y, poison_rate, seed):
    X, y = X.copy(), y.copy()
    rng = np.random.default_rng(seed)
    spoofed_idx = np.where(y == 1)[0]
    chosen = rng.choice(spoofed_idx, size=int(len(spoofed_idx) * poison_rate), replace=False)
    X[chosen, CN0_IDX] = TRIGGER_VALUE_SCALED
    y[chosen] = 0
    return X, y, len(chosen)

c5 = clients[4]
X_tr_p, y_tr_p, n_tr   = poison_data(c5['X_tr'],  c5['y_tr'],  POISON_RATE, SEED)
X_val_p, y_val_p, n_val = poison_data(c5['X_val'], c5['y_val'], POISON_RATE, SEED + 1)
c5_poison = {'X_tr': X_tr_p, 'y_tr': y_tr_p, 'X_val': X_val_p, 'y_val': y_val_p}
poisoned_clients = clients[:4] + [c5_poison]

print(f'Client 5 poisoned: {n_tr} train rows, {n_val} val rows')

Client 5 poisoned: 1632 train rows, 288 val rows


In [7]:
# Cole — clean and triggered test sets (same as week07)
X_clean_test = X_test_sc.copy()
y_clean_test = y_test.copy()

spoofed_test_mask = (y_test == 1)
X_triggered = X_test_sc[spoofed_test_mask].copy()
y_triggered  = y_test[spoofed_test_mask].copy()
X_triggered[:, CN0_IDX] = TRIGGER_VALUE_SCALED

print(f'Clean test: {len(X_clean_test)} rows')
print(f'Triggered test: {len(X_triggered)} rows (spoofed, trigger applied, true label kept)')

Clean test: 15000 rows
Triggered test: 6000 rows (spoofed, trigger applied, true label kept)


## 3. Challenge Set Construction (D5 — new in Week 8)

The aggregator constructs a **CN0-boundary challenge set**: rows from the server-side clean test data where CN0 falls in the ambiguous overlap zone between the spoofed and benign distributions.

- Lower bound: spoofed training CN0 75th percentile (upper tail of the spoofed distribution)
- Upper bound: benign training CN0 25th percentile (lower tail of the benign distribution)

This is the zone where a backdoored model (which learned "high CN0 → predict benign") will generalize its backdoor behavior and misclassify spoofed samples. An honest model will still use the other 9 features to classify correctly.

The server does **not** need to know the specific trigger value (46.718). It only needs to know that CN0 is the discriminating feature — domain knowledge that is already documented in the paper.

In [8]:
# Build the CN0-boundary challenge set from the server's clean test data.
#
# The challenge set is: spoofed test rows whose CN0 (unscaled) is ABOVE the
# spoofed training 75th percentile. These are high-CN0 spoofed samples that sit
# in the trigger region. A backdoored model generalises 'high CN0 -> predict benign'
# and misclassifies them; an honest model uses the other 9 features and keeps them
# as spoofed. Challenge accuracy is therefore LOW for Client 5 and HIGH for honest
# clients -- exactly the signal we need for detection.
#
# The server does NOT need to know the exact trigger value (46.718). It only needs
# to know CN0 is the discriminating feature (domain knowledge already in the paper).
cn0_test_unscaled = X_test_sc[:, CN0_IDX] * scaler.scale_[CN0_IDX] + scaler.mean_[CN0_IDX]

# Keep only spoofed test rows with CN0 above the spoofed training 75th percentile
challenge_mask = (y_test == 1) & (cn0_test_unscaled >= cn0_spoofed_75th)
X_challenge = X_test_sc[challenge_mask]
y_challenge = y_test[challenge_mask]   # all 1s — true label is spoofed

print(f'Challenge set: {len(X_challenge)} spoofed rows with CN0 >= {cn0_spoofed_75th:.3f} (spoofed 75th pct)')
print(f'  CN0 range in challenge set: [{cn0_test_unscaled[challenge_mask].min():.3f}, {cn0_test_unscaled[challenge_mask].max():.3f}]')
print(f'  All labels spoofed: {(y_challenge == 1).all()}')
print(f'  Honest models should score HIGH here; backdoored model should score LOW.')

Challenge set: 1502 spoofed rows with CN0 >= 46.181 (spoofed 75th pct)
  CN0 range in challenge set: [46.181, 50.339]
  All labels spoofed: True
  Honest models should score HIGH here; backdoored model should score LOW.


## 4. Model Definition and Helpers

In [9]:
# Dilpreet — model and FL helpers (same as week07)
class BinaryDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32),        nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(32, 16),        nn.ReLU(),
            nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

INPUT_DIM = len(FEATURES)

def make_loader(X, y, batch_size=512, shuffle=True):
    ds = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(y.astype(np.float32)))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

def train_local(model, X_tr, y_tr, X_val, y_val, epochs=3, lr=1e-3, device=DEVICE):
    loader = make_loader(X_tr, y_tr)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()
    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
    return eval_acc(model, X_val, y_val, device)

def eval_acc(model, X, y, device=DEVICE):
    model.eval()
    with torch.no_grad():
        preds = (model(torch.FloatTensor(X).to(device)).cpu() > 0).long().numpy()
    return (preds == y).mean()

def eval_preds(model, X, device=DEVICE):
    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X).to(device)).cpu()
    return (logits > 0).long().numpy()

def get_params(model):
    return [p.data.clone() for p in model.parameters()]

def set_params(model, params):
    for p, v in zip(model.parameters(), params):
        p.data.copy_(v)

def fedavg(param_list, weights=None):
    if weights is None:
        weights = [1.0 / len(param_list)] * len(param_list)
    return [sum(w * p for w, p in zip(weights, layers)) for layers in zip(*param_list)]

def coord_median(param_list):
    """Coordinate-wise median aggregation (D3).
    For each weight tensor, take the element-wise median across all clients.
    A single outlier client cannot shift the median far."""
    result = []
    for layers in zip(*param_list):
        stacked = torch.stack(list(layers), dim=0)  # [n_clients, *param_shape]
        result.append(stacked.median(dim=0).values)
    return result

def report(label, model, X_clean, y_clean, X_trig, y_trig):
    clean_acc = eval_acc(model, X_clean, y_clean)
    bdr = (eval_preds(model, X_trig) == 0).mean()
    print(f'[{label}]  clean acc={clean_acc:.4f}  BSR={bdr:.4f}')
    return clean_acc, bdr

print('Helpers defined (includes coord_median for D3).')

Helpers defined (includes coord_median for D3).


## 5. Experiments

### Exp A — Attack Baseline (no defense, reproduces Week 7 Exp 3)
Sanity check: same poisoned FedAvg setup as before. Should match Week 7 result: BSR ≈ 74.35%.

In [10]:
# Exp A — poisoned FedAvg, no defense (week07 reproduction)
FL_ROUNDS    = 10
LOCAL_EPOCHS = 3

global_A = BinaryDNN(INPUT_DIM).to(DEVICE)

for rnd in range(FL_ROUNDS):
    local_params = []
    for c in poisoned_clients:
        local_m = copy.deepcopy(global_A)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
    set_params(global_A, fedavg(local_params))

acc_A, bdr_A = report('Exp A: Poisoned, no defense', global_A, X_clean_test, y_clean_test, X_triggered, y_triggered)
print(f'  (Week 7 reference: clean acc=0.7003, BSR=0.7435)')

[Exp A: Poisoned, no defense]  clean acc=0.6978  BSR=0.7553
  (Week 7 reference: clean acc=0.7003, BSR=0.7435)


### Exp B — D5 Trigger-Region Probing Only

Each round:
1. Each client's submitted model is evaluated on the CN0-boundary challenge set
2. Clients >1.5 SD below the group median challenge accuracy are flagged
3. Flagged clients' weight deltas are clipped to 10% before aggregation
4. Aggregation still uses mean (no D3 yet)

Expected: BSR drops meaningfully. Client 5 should be flagged most rounds.

In [11]:
# Exp B — D5 probing + clipping, mean aggregation
CLIP_FACTOR  = 0.10   # flagged client's delta reduced to 10%
FLAG_SIGMA   = 1.5    # flag if challenge acc < median - 1.5 * std

global_B = BinaryDNN(INPUT_DIM).to(DEVICE)
flag_log_B = []  # track which clients get flagged each round

for rnd in range(FL_ROUNDS):
    local_params     = []
    challenge_accs   = []
    global_params_now = get_params(global_B)

    for i, c in enumerate(poisoned_clients):
        local_m = copy.deepcopy(global_B)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
        # D5: evaluate on challenge set (server does this, client has no say)
        ch_acc = eval_acc(local_m, X_challenge, y_challenge)
        challenge_accs.append(ch_acc)

    # Flagging: >1.5 SD below group median
    ch_arr    = np.array(challenge_accs)
    ch_median = np.median(ch_arr)
    ch_std    = ch_arr.std()
    threshold = ch_median - FLAG_SIGMA * ch_std
    flagged   = ch_arr < threshold
    flag_log_B.append(flagged.copy())

    # Clip flagged clients: replace their full update with a 10% step toward it
    processed_params = []
    for i, params in enumerate(local_params):
        if flagged[i]:
            # clipped_params = global + CLIP_FACTOR * (local - global)
            clipped = [g + CLIP_FACTOR * (p - g)
                       for g, p in zip(global_params_now, params)]
            processed_params.append(clipped)
        else:
            processed_params.append(params)

    set_params(global_B, fedavg(processed_params))

    if rnd == 0 or rnd == FL_ROUNDS - 1:
        flagged_clients = [f'C{i+1}' for i, f in enumerate(flagged) if f]
        print(f'Round {rnd+1}: challenge accs={[f"{a:.3f}" for a in ch_arr]}  '
              f'threshold={threshold:.3f}  flagged={flagged_clients if flagged_clients else "none"}')

acc_B, bdr_B = report('Exp B: D5 probing only', global_B, X_clean_test, y_clean_test, X_triggered, y_triggered)

# Summarise flagging
flag_arr_B = np.array(flag_log_B)  # [FL_ROUNDS, N_CLIENTS]
print(f'\nClient 5 flagged in {flag_arr_B[:, 4].sum()}/{FL_ROUNDS} rounds')
print(f'False positives (honest clients flagged at least once): '
      f'{[(i+1) for i in range(4) if flag_arr_B[:, i].any()]}')

Round 1: challenge accs=['0.000', '0.000', '0.003', '0.000', '0.000']  threshold=-0.002  flagged=none


Round 10: challenge accs=['0.413', '0.421', '0.465', '0.477', '0.001']  threshold=0.153  flagged=['C5']
[Exp B: D5 probing only]  clean acc=0.6975  BSR=0.6725

Client 5 flagged in 9/10 rounds
False positives (honest clients flagged at least once): []


### Exp C — D5 Probing + D3 Coordinate-Wise Median

Same as Exp B, but instead of averaging the (clipped) client updates with mean, we use **coordinate-wise median** across all clients. This provides a second layer: even if a slightly poisoned update slips past the probing threshold, the median suppresses its influence parameter-by-parameter.

In [12]:
# Exp C — D5 probing + D3 coordinate-wise median
global_C = BinaryDNN(INPUT_DIM).to(DEVICE)
flag_log_C = []

for rnd in range(FL_ROUNDS):
    local_params      = []
    challenge_accs    = []
    global_params_now = get_params(global_C)

    for i, c in enumerate(poisoned_clients):
        local_m = copy.deepcopy(global_C)
        train_local(local_m, c['X_tr'], c['y_tr'], c['X_val'], c['y_val'], epochs=LOCAL_EPOCHS)
        local_params.append(get_params(local_m))
        ch_acc = eval_acc(local_m, X_challenge, y_challenge)
        challenge_accs.append(ch_acc)

    ch_arr    = np.array(challenge_accs)
    ch_median = np.median(ch_arr)
    ch_std    = ch_arr.std()
    threshold = ch_median - FLAG_SIGMA * ch_std
    flagged   = ch_arr < threshold
    flag_log_C.append(flagged.copy())

    # Clip flagged clients (same as Exp B)
    processed_params = []
    for i, params in enumerate(local_params):
        if flagged[i]:
            clipped = [g + CLIP_FACTOR * (p - g)
                       for g, p in zip(global_params_now, params)]
            processed_params.append(clipped)
        else:
            processed_params.append(params)

    # D3: coordinate-wise median instead of mean
    set_params(global_C, coord_median(processed_params))

    if rnd == 0 or rnd == FL_ROUNDS - 1:
        flagged_clients = [f'C{i+1}' for i, f in enumerate(flagged) if f]
        print(f'Round {rnd+1}: challenge accs={[f"{a:.3f}" for a in ch_arr]}  '
              f'threshold={threshold:.3f}  flagged={flagged_clients if flagged_clients else "none"}')

acc_C, bdr_C = report('Exp C: D5 + D3 (full defense)', global_C, X_clean_test, y_clean_test, X_triggered, y_triggered)

flag_arr_C = np.array(flag_log_C)
print(f'\nClient 5 flagged in {flag_arr_C[:, 4].sum()}/{FL_ROUNDS} rounds')
print(f'False positives: {[(i+1) for i in range(4) if flag_arr_C[:, i].any()]}')

Round 1: challenge accs=['0.021', '0.013', '0.033', '0.047', '0.000']  threshold=-0.003  flagged=none


Round 10: challenge accs=['0.571', '0.489', '0.534', '0.489', '0.003']  threshold=0.175  flagged=['C5']
[Exp C: D5 + D3 (full defense)]  clean acc=0.7067  BSR=0.5203

Client 5 flagged in 9/10 rounds
False positives: []


## 6. Results Summary

In [13]:
# Week 7 reference values (from executed notebook)
bdr_baseline_wk7   = 0.4802  # centralized baseline
bdr_fedavg_honest  = 0.5208  # honest FedAvg (upper bound we want to approach)
bdr_fedavg_poison  = 0.7435  # Week 7 Exp 3 (what Exp A should match)

results = pd.DataFrame([
    {'Experiment': 'Wk7 centralized baseline (reference)',  'Clean Acc': 0.7418, 'BSR': bdr_baseline_wk7,  'Lift': 0.0000},
    {'Experiment': 'Wk7 FedAvg honest (target BSR)',        'Clean Acc': 0.7257, 'BSR': bdr_fedavg_honest, 'Lift': bdr_fedavg_honest - bdr_baseline_wk7},
    {'Experiment': 'Wk7 FedAvg poisoned (attack, no defense)', 'Clean Acc': 0.7003, 'BSR': bdr_fedavg_poison, 'Lift': bdr_fedavg_poison - bdr_baseline_wk7},
    {'Experiment': 'Exp A: Poisoned, no defense (repro)',   'Clean Acc': acc_A,  'BSR': bdr_A,  'Lift': bdr_A - bdr_baseline_wk7},
    {'Experiment': 'Exp B: + D5 probing only',              'Clean Acc': acc_B,  'BSR': bdr_B,  'Lift': bdr_B - bdr_baseline_wk7},
    {'Experiment': 'Exp C: + D5 probing + D3 median',       'Clean Acc': acc_C,  'BSR': bdr_C,  'Lift': bdr_C - bdr_baseline_wk7},
])

print(results.to_string(index=False))
print()

bsr_reduction_B = bdr_A - bdr_B
bsr_reduction_C = bdr_A - bdr_C
print(f'D5 only (Exp B vs A):       BSR reduction = {bsr_reduction_B:+.4f}')
print(f'D5+D3  (Exp C vs A):        BSR reduction = {bsr_reduction_C:+.4f}')
print(f'Clean acc cost (Exp C vs A): {acc_C - acc_A:+.4f}')
print()
print(f'Target BSR (honest FedAvg baseline): {bdr_fedavg_honest:.4f}')
print(f'Defense closed the gap by:  {(bdr_A - bdr_C) / (bdr_A - bdr_fedavg_honest) * 100:.1f}%  (Exp C)')

                              Experiment  Clean Acc      BSR     Lift
    Wk7 centralized baseline (reference)   0.741800 0.480200 0.000000
          Wk7 FedAvg honest (target BSR)   0.725700 0.520800 0.040600
Wk7 FedAvg poisoned (attack, no defense)   0.700300 0.743500 0.263300
     Exp A: Poisoned, no defense (repro)   0.697800 0.755333 0.275133
                Exp B: + D5 probing only   0.697533 0.672500 0.192300
         Exp C: + D5 probing + D3 median   0.706733 0.520333 0.040133

D5 only (Exp B vs A):       BSR reduction = +0.0828
D5+D3  (Exp C vs A):        BSR reduction = +0.2350
Clean acc cost (Exp C vs A): +0.0089

Target BSR (honest FedAvg baseline): 0.5208
Defense closed the gap by:  100.2%  (Exp C)


In [14]:
# Per-round flagging detail for the paper / presentation
print('=== Exp B — per-round flagging (D5 only) ===')
for rnd, flags in enumerate(flag_log_B):
    fc = [f'C{i+1}' for i, f in enumerate(flags) if f]
    print(f'  Round {rnd+1:2d}: flagged = {fc if fc else "none"}')

print()
print('=== Exp C — per-round flagging (D5 + D3) ===')
for rnd, flags in enumerate(flag_log_C):
    fc = [f'C{i+1}' for i, f in enumerate(flags) if f]
    print(f'  Round {rnd+1:2d}: flagged = {fc if fc else "none"}')

=== Exp B — per-round flagging (D5 only) ===
  Round  1: flagged = none
  Round  2: flagged = ['C5']
  Round  3: flagged = ['C5']
  Round  4: flagged = ['C5']
  Round  5: flagged = ['C5']
  Round  6: flagged = ['C5']
  Round  7: flagged = ['C5']
  Round  8: flagged = ['C5']
  Round  9: flagged = ['C5']
  Round 10: flagged = ['C5']

=== Exp C — per-round flagging (D5 + D3) ===
  Round  1: flagged = none
  Round  2: flagged = ['C5']
  Round  3: flagged = ['C5']
  Round  4: flagged = ['C5']
  Round  5: flagged = ['C5']
  Round  6: flagged = ['C5']
  Round  7: flagged = ['C5']
  Round  8: flagged = ['C5']
  Round  9: flagged = ['C5']
  Round 10: flagged = ['C5']
